# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal, créez l'environnement avec `uv init`, `uv venv`, puis activez-le avec `.venv\Scripts\activate`. Ajoutez les dépendances avec `uv add ipykernel langchain langchain-google-genai langchain-text-splitters langchain-community langchain-chroma python-dotenv pypdf chromadb tiktoken`. Placez votre clé dans `.env` sous la forme `GEMINI_API_KEY=votre_cle` et ne publiez jamais ce fichier. Les embeddings et les appels de génération envoient le texte du CV à l'API Gemini ; ne poursuivez que si vous acceptez ce traitement distant.


In [52]:
import json
import os
from pathlib import Path

import tiktoken
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_chroma import Chroma


##  LLM

In [53]:
dotenv_candidates = [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env",
    Path.cwd() / "TP4" / "RAG" / ".env",
]
dotenv_path = next((candidate for candidate in dotenv_candidates if candidate.is_file()), None)
env_loaded = load_dotenv(dotenv_path=dotenv_path, override=True) if dotenv_path else False
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
print("Fichier .env chargé :", bool(env_loaded))
print("GEMINI_API_KEY détectée :", bool(GEMINI_API_KEY))


Fichier .env chargé : True
GEMINI_API_KEY détectée : True


In [54]:
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
llm = None
if not GEMINI_API_KEY:
    print("Client Gemini non créé : vérifiez GEMINI_API_KEY dans le fichier .env.")
else:
    try:
        llm = ChatGoogleGenerativeAI(
            model="gemini-3.8-flash",
            temperature=0,
            api_key=GEMINI_API_KEY,
        )
        print("Client Gemini prêt.")
    except Exception as exc:
        print(f"Initialisation Gemini indisponible ({type(exc).__name__}) : {exc}")


def response_text(response):
    """Return only textual content from a LangChain/Gemini response."""
    content = getattr(response, "content", response)
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, (list, tuple)):
        parts = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict) and isinstance(block.get("text"), str):
                parts.append(block["text"])
        return "\n".join(part for part in parts if part).strip()
    return "" if content is None else str(content).strip()


def safe_invoke(model, content):
    if model is None:
        print("Appel ignoré : le client Gemini n’est pas disponible.")
        return None
    try:
        text = response_text(model.invoke(content))
        return text if text else None
    except Exception as exc:
        print(f"Appel Gemini indisponible ({type(exc).__name__}) : {exc}")
        return None


Client Gemini prêt.


# 1. Loading the pdf, Chunking

Le document utilisé pour ce TP est un CV connu, placé dans le dossier `pdfs`. Les informations personnelles restent utilisées localement et ne doivent pas être publiées dans un dépôt public.


In [55]:
pdf_file="./pdfs/cv_ahmed_miloudi.pdf"

In [56]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [57]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [58]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


2

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [59]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = None
if not GEMINI_API_KEY:
    print("Embeddings non initialisés : vérifiez GEMINI_API_KEY dans le fichier .env.")
else:
    try:
        embedding_model = GoogleGenerativeAIEmbeddings(
            model="gemini-embedding-001",
            google_api_key=GEMINI_API_KEY,
        )
        print("Modèle d’embeddings Gemini prêt.")
    except Exception as exc:
        print(f"Initialisation des embeddings indisponible ({type(exc).__name__}) : {exc}")


Modèle d’embeddings Gemini prêt.


In [60]:
if embedding_model is None:
    print("Test d’embedding ignoré : le modèle n’est pas initialisé.")
else:
    try:
        test_embedding = embedding_model.embed_query("compétences techniques")
        print("Test d’embedding réussi ; dimension :", len(test_embedding))
    except Exception as exc:
        print(f"Test d’embedding indisponible ({type(exc).__name__}) : {exc}")


Test d’embedding réussi ; dimension : 3072


Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

In [61]:
vectorstore = None
if embedding_model is None:
    print("Base vectorielle non créée : le modèle d’embeddings Gemini est indisponible.")
else:
    try:
        vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=embedding_model,
            ids=[f"cv-chunk-{i}" for i in range(len(chunks))],
            collection_name="cv_ahmed_miloudi_gemini",
            persist_directory="./store_gemini",
        )
        print(f"Base vectorielle prête : {len(chunks)} chunks indexés.")
    except Exception as exc:
        print(f"Indexation Gemini indisponible ({type(exc).__name__}) : {exc}")


Base vectorielle prête : 2 chunks indexés.


# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [62]:
retriever = None
if vectorstore is not None:
    retriever = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5},
    )
    print("Retriever prêt (5 chunks maximum).")
else:
    print("Retriever indisponible : relancez l’indexation après correction de la clé ou du quota.")


Retriever prêt (5 chunks maximum).


On va faire un test. J'ai la question suivante : 
> « Quelles sont les compétences techniques présentées dans le CV ? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [63]:
if retriever is None:
    print("Recherche ignorée : le retriever n’est pas prêt.")
else:
    try:
        retrieved_chunks = retriever.invoke(
            "Quelles sont les compétences techniques présentées dans le CV ?"
        )
        print(retrieved_chunks)
        print("Nombre de chunks :", len(retrieved_chunks))
    except Exception as exc:
        print(f"Recherche indisponible ({type(exc).__name__}) : {exc}")


[Document(id='cv-chunk-1', metadata={'creationdate': '2024-11-13T23:34:50+00:00', 'total_pages': 1, 'author': 'Ahmed Miloudi', 'producer': 'Canva', 'page_label': '1', 'source': './pdfs/cv_ahmed_miloudi.pdf', 'title': 'White simple student cv resume', 'page': 0, 'moddate': '2024-11-13T23:34:49+00:00', 'keywords': 'DAFktloEDg4,BAFa1r9oSJM', 'creator': 'Canva'}, page_content='Rigueur\nAutonomie \nCapacité à gérer les tâches de manière indépendante\nOrientation client forte, avec une attention particulière à\nla satisfaction et aux besoins\nSens du service client\nTravail en équipe\nSolides compétences en collaboration et aisance à\ns’adapter au sein d’équipes multidisciplinaires.\n2024     DEUST en MIPC \n2024     TCF B1 ( score global : 362 )\n2022     Baccalauréat scientifique (Mention Bien)'), Document(id='cv-chunk-0', metadata={'creationdate': '2024-11-13T23:34:50+00:00', 'author': 'Ahmed Miloudi', 'page_label': '1', 'title': 'White simple student cv resume', 'total_pages': 1, 'moddat

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [64]:
prompt_template="""
Answer the following question based only on the provided CV context.
The context contains the candidate's profile, education and skills.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Le document utilisé est un CV. Vous pouvez préciser dans le prompt que le contexte concerne le parcours, la formation et les compétences présentés dans ce CV.

Le contexte est délimité par la balise `<context>`, et la question par la balise `<question>`.

### Gestion des hallucinations
Pour éviter les hallucinations, spécifiez dans le prompt : *« Si la réponse ne figure pas dans ce contexte, répondez par : "Je ne sais pas !" »*. Cela permet de régler efficacement le problème des hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [65]:
user_input = "Quelles sont les compétences techniques présentées dans le CV ?"
context_for_query = ""
if retriever is None:
    print("Contexte indisponible : le retriever n’est pas prêt.")
else:
    try:
        relevant_document_chunks = retriever.invoke(user_input)
        context_list = [doc.page_content for doc in relevant_document_chunks]
        context_for_query = ". ".join(context_list)
        print(context_for_query)
    except Exception as exc:
        print(f"Récupération du contexte indisponible ({type(exc).__name__}) : {exc}")


Rigueur
Autonomie 
Capacité à gérer les tâches de manière indépendante
Orientation client forte, avec une attention particulière à
la satisfaction et aux besoins
Sens du service client
Travail en équipe
Solides compétences en collaboration et aisance à
s’adapter au sein d’équipes multidisciplinaires.
2024     DEUST en MIPC 
2024     TCF B1 ( score global : 362 )
2022     Baccalauréat scientifique (Mention Bien). F 93 TRANCHE 6 TAMANSOURT
+212 612 675 763
2bactawjih2022@gmail.com
ETUDIANT
Étudiant en informatique,
passionné et motivé, avec
une forte envie d'appliquer
mes compétences
techniques et de développer
de nouvelles expertises.
Ouvert à relever des défis et
à contribuer à des projets
innovants dans le domaine
de l'informatique.
Connaissances scolaires
Maîtrise des Langues
Français 
Anglais
Arabe
Maîtrise avancée 
Langue maternelle
AHMED
MILOUDI
PROFILE
FORMATION
2024-2025
Faculté de Science et Technique, Marrakech 
Licence en Système Informatique et Réparti
2021-2022 Baccalauréat

In [66]:
len(relevant_document_chunks)

2

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [67]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on the provided CV context.
The context contains the candidate's profile, education and skills.
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
Rigueur
Autonomie 
Capacité à gérer les tâches de manière indépendante
Orientation client forte, avec une attention particulière à
la satisfaction et aux besoins
Sens du service client
Travail en équipe
Solides compétences en collaboration et aisance à
s’adapter au sein d’équipes multidisciplinaires.
2024     DEUST en MIPC 
2024     TCF B1 ( score global : 362 )
2022     Baccalauréat scientifique (Mention Bien). F 93 TRANCHE 6 TAMANSOURT
+212 612 675 763
2bactawjih2022@gmail.com
ETUDIANT
Étudiant en informatique,
passionné et motivé, avec
une forte envie d'appliquer
mes compétences
techniques et de développer
de nouvelles expertises.
Ouvert à relever des défis et
à contribuer à des pro

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [68]:
# Le notebook utilise le client Gemini initialisé dans la cellule de configuration.
if llm is None:
    print("Modèle Gemini indisponible : vérifiez la clé et les cellules de configuration.")
else:
    print("Modèle Gemini prêt pour la génération RAG.")


Modèle Gemini prêt pour la génération RAG.


In [69]:
from IPython.display import Markdown, display

resp = safe_invoke(llm, prompt)
if resp is not None:
    display(Markdown(resp))
else:
    print("Aucune réponse Gemini disponible ; vérifiez la connexion, la clé et le quota.")


D'après le CV, les compétences techniques présentées sont :

* **Développement web :** Création de sites web avec WordPress et HTML & CSS
* **Systèmes d'exploitation :** Maîtrise de Windows et Linux
* **Bases de données :** Connaissance approfondie de SQL
* **Réseaux informatiques :** Configuration et gestion des réseaux

## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [70]:
def RAG(query, model=None, template=prompt_template):
    active_model = model if model is not None else globals().get("llm")
    active_retriever = globals().get("retriever")
    if active_retriever is None:
        print("RAG indisponible : le retriever n’est pas prêt.")
        return None
    try:
        context_docs = active_retriever.invoke(query)
        context = ". ".join(doc.page_content for doc in context_docs)
        formatted_prompt = template.format(context=context, question=query)
        response = safe_invoke(active_model, formatted_prompt)
        return response
    except Exception as exc:
        print(f"Exécution RAG indisponible ({type(exc).__name__}) : {exc}")
        return None


### Appel de RAG 

In [71]:
resp = RAG("Quelles formations sont mentionnées dans le CV ?")
if resp is not None:
    display(Markdown(resp))
else:
    print("Aucune réponse RAG disponible pour le moment.")


Basé sur le CV fourni, les formations mentionnées sont :

* **2024-2025 :** Licence en Système Informatique et Réparti – Faculté de Science et Technique, Marrakech.
* **2021-2022 :** Baccalauréat Scientifique – Lycée Qualifiant Ryad Ezzahia.

*(Il est également fait mention dans les diplômes/certificats d'un **DEUST en MIPC** en 2024 ainsi que du **Baccalauréat scientifique (Mention Bien)** en 2022).*

# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [72]:
#question ou query 
user_input="Quelles compétences techniques sont listées dans le CV ?"

Récupérer le context 

In [73]:
context_for_query = ""
if retriever is not None:
    try:
        relevant_document_chunks = retriever.invoke(user_input)
        context_list = [doc.page_content for doc in relevant_document_chunks]
        context_for_query = ". ".join(context_list)
    except Exception as exc:
        print(f"Récupération du contexte indisponible ({type(exc).__name__}) : {exc}")
else:
    print("Contexte indisponible : le retriever n’est pas prêt.")


In [74]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [75]:
answer = RAG("Quelles compétences techniques sont listées dans le CV ?")
if answer is not None:
    display(Markdown(answer))
else:
    print("Aucune réponse RAG disponible pour le moment.")


D'après le CV, les compétences techniques listées sont :

* **Développement web :** Création de sites web avec WordPress et HTML & CSS
* **Systèmes d'exploitation :** Maîtrise de Windows et Linux
* **Bases de données :** Connaissance approfondie de SQL
* **Réseaux informatiques :** Configuration et gestion des réseaux

## 7.2- *Groundness*

Le prompt pour le juge 

In [76]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [77]:
# Le juge réutilise le client Gemini configuré plus haut, sans seconde clé API.
groundness_checker = llm
if groundness_checker is None:
    print("Évaluateur indisponible : configurez la clé Gemini avant cette section.")
else:
    print("Évaluateur Gemini prêt ; l’évaluation consommera un appel API supplémentaire.")


Évaluateur Gemini prêt ; l’évaluation consommera un appel API supplémentaire.


In [78]:
groundness_checker = llm

def evaluate(system_message, user_message_template, question, model=None):
    active_retriever = globals().get("retriever")
    active_model = model if model is not None else groundness_checker
    if active_retriever is None:
        print("Évaluation ignorée : le retriever n’est pas prêt.")
        return None
    try:
        retrieved_chunks = active_retriever.invoke(question)
        context = ". ".join(doc.page_content for doc in retrieved_chunks)
        answer = RAG(question)
        if answer is None:
            print("Évaluation ignorée : aucune réponse RAG à noter.")
            return None
        prompt = (
            f"{system_message}\nUSER:\n"
            f"{user_message_template.format(question=question, context=context, answer=answer)}"
        )
        judge_response = safe_invoke(active_model, prompt)
        return judge_response
    except Exception as exc:
        print(f"Évaluation indisponible ({type(exc).__name__}) : {exc}")
        return None


In [79]:
resp = evaluate(groundness_rater_system_message, user_message_template, user_input)
if resp is not None:
    display(Markdown(resp))
else:
    print("Évaluation non disponible ; vérifiez la clé, le quota et les cellules précédentes.")


### Étapes nécessaires pour évaluer la réponse selon la métrique

1. **Identifier les faits et informations clés fournis dans la réponse** : Relever chaque élément mentionné par l'IA (catégories et détails des compétences techniques).
2. **Vérifier la présence de ces informations dans le contexte** : Comparer chaque point de la réponse avec le texte source fourni dans `###Context`.
3. **Détecter les éventuelles extrapolations ou hallucinations** : S'assurer qu'aucune information extérieure au texte n'a été ajoutée.
4. **Conclure et attribuer une note** : Attribuer un score de 1 à 5 en fonction du respect de la métrique.

---

### Évaluation étape par étape

1. **Analyse de la section "Développement web"** :
   - *Réponse* : « Développement web : Création de sites web avec WordPress et HTML & CSS »
   - *Contexte* : On retrouve textuellement sous la rubrique *Compétences Techniques* : « Création de sites web avec WordPress et HTML& CSS » et « Développement web ».
2. **Analyse de la section "Systèmes d'exploitation"** :
   - *Réponse* : « Systèmes d'exploitation : Maîtrise de Windows et Linux »
   - *Contexte* : Présent textuellement dans le contexte : « Systèmes d'exploitation » et « Maîtrise de Windows et Linux ».
3. **Analyse de la section "Bases de données"** :
   - *Réponse* : « Bases de données : Connaissance approfondie de SQL »
   - *Contexte* : Présent textuellement dans le contexte : « Bases de données » et « Connaissance approfondie de SQL ».
4. **Analyse de la section "Réseaux informatiques"** :
   - *Réponse* : « Réseaux informatiques : Configuration et gestion des réseaux »
   - *Contexte* : Présent textuellement dans le contexte : « Réseaux informatiques » et « Configuration et gestion des réseaux ».

Toutes les informations contenues dans la réponse proviennent fidèlement et exclusivement de la section "Compétences Techniques" du document fourni, sans ajout d'informations externes.

---

### Degré de respect de la métrique

La métrique est entièrement respectée.

---

### Score

**5** (La métrique est entièrement respectée)

### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
